# 概要
- 引継ぎのためのnotebook
- 生ログデータを加工し、モデルの訓練/テストを実行するまでの手順の共有を意図する
- cliから動かしてもよいけど、基本的にjupyter-notebookから動かすことを想定してます

データについて
- data/ に格納されている（実体は~/dataにある。ここへバケットをマウントしている。）  
起動時に次のコマンドで都度マウントすること：
> s3fs amiya-iwamura /home/ubuntu/data -o passwd_file=${HOME}/.passwd-s3fs -o url=https://s3.isk01.sakurastorage.jp/ -o use_path_request_style -o nomultipart

- raw：生ログデータ
- interim：Evtxを変換したcsvなど
- processed：モデル前データ

# Import

In [1]:
from pathlib import Path
import pandas as pd
import os
import numpy as np
from tqdm import tqdm
from xml.etree.ElementTree import fromstring, ElementTree
from Evtx.Evtx import Evtx
import csv

In [2]:
%load_ext autoreload
%autoreload 2
import preprocess 
import main 
import visualize_utils
from unmask_attributes import process_csv
import embedding_converter

/home/ubuntu/My_lad/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
INTERIM_DIR = Path('../data/interim')
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
NO_MEANING_DIR = Path('../data/no_meaning')
PROJECT_ROOT = Path("../")

# 実行例１

## データ前処理
- シナリオT1105を例に取り、手順を示す

In [4]:
parent_dir = "ScenarioData"
project_name = 'T1105'

### evtx ⇒ csv
- Evtx下のsecurity.evtxを入力として、これをフラット化したsecurity.csvを出力する

In [6]:
# 少数サンプルで正常に実行が行われるかテスト
input_dir = RAW_DIR/parent_dir/project_name
output_dir = INTERIM_DIR/project_name

preprocess.evtx_to_csv_without_eventdata_columns_samplingver(
    evtx_filepath = input_dir/"20240927120753_7EA74D56-6663-313B-2CC1-A7843FCD1AE6/Evtx/Security.evtx",
    output_dir = output_dir,
    output_filename = "security_test",
)

Writing to CSV: 100%|██████████| 50/50 [00:00<00:00, 50951.21it/s]


In [ ]:
# 実際にフルサイズに対して実行
preprocess.evtx_to_csv_without_eventdata_columns(
    evtx_filepath = input_dir/"Evtx/Security.evtx",
    output_dir = output_dir,
    output_filename = "security",
)

### アノテーション
- 各ログについて正常/異常のラベル付与

In [ ]:
# アノテーション用のデータフレームを作成
ano = pd.read_excel(RAW_DIR/parent_dir/"_anotation/T1055/T1055_20250304.xlsx")
df = pd.DataFrame()
df["Channel"] = ano["Unnamed: 9"].iloc[1:].values
df["EventRecordID"] = ano["Unnamed: 14"].iloc[1:].values

df = df[df["Channel"] == "Sec"]

df["EventRecordID"] = df["EventRecordID"].astype(int)

# csvに反映する
# 既存ファイルを上書きする操作であることに注意！！！！！！！
labeled = preprocess.anotate_csv(
    csv_filepath = INTERIM_DIR/project_name/"security.csv",
    ano_df = df,
    output_dir = INTERIM_DIR/project_name,
)

### パース
- Drain を用いたログの構造化

In [ ]:
input_dir = INTERIM_DIR/project_name
output_dir = INTERIM_DIR/project_name
preprocess.parse_log(input_dir=input_dir, output_dir=output_dir, logfile_name='security', parser_type="drain")

## モデル前データ作成
- 作成先のディレクトリ名を「recover」と設定
- processed/recover へモデル前データ(T1105/ & train)が作成される
- T1105/ がテストデータ、trainが訓練データに対応する
- T1105/ にはテストデータにおける正常onlyデータと異常onlyデータの2種が格納されている

In [ ]:
output_dirname = "recover"
hash_method = "md5" # md5とlshが実装済み

eventid = f"EventTemplateUnmaskedHash_{hash_method}"
project = "T1105"
train_ratio = 0.94

# 訓練データ作成
preprocess.prepare_model_train_data(
    logdata_filepath = INTERIM_DIR / f"{project}/security_structured_unmasked.csv",
    output_dir = PROCESSED_DIR / output_dirname / project,
    train_ratio=train_ratio,
    features = [f"{eventid}", "deltaT"],
    use_columns = ["timestamp", "Label", "EventID", f"{eventid}", "deltaT"],
    window_size = 430,
    step_size = 200,
    mode = "fixed", 
)

# テストデータ作成
preprocess.prepare_model_test_data(
    logdata_filepath = INTERIM_DIR / f"{project}/security_structured_unmasked.csv",
    output_dir = PROCESSED_DIR / output_dirname / project,
    train_ratio=train_ratio,
    features = [f"{eventid}", "deltaT"],
    use_columns = ["timestamp", "Label", "EventID", f"{eventid}", "deltaT"],
    window_size = 430,
    step_size = 200,
    mode = "fixed", 
)

## モデル訓練＆テスト
- main.pyのmain_cilから実行する
- conf/bert以下のconfigファイルを指定し、これに従って実行される（事前に準備しておくこと）

#### 訓練
- 第1引数：実行モード（train or test）
- 第2引数：configファイルパス
- 第3引数以降：設定の上書き（configファイルでロードした設定を別途上書き）

In [ ]:
# train
main.main_cli([
    "train",
    "bert/recover",           # conf/bert_config_name.yaml
    #"default.device_id=0",        
    #"default.epochs=10",　　　# ここに好きな key=value を並べる
])

#### テスト
- 第１引数：実行モード（train or test）
- 第2引数：訓練済みモデルの重みファイルパス
- 第3引数以降：設定の上書き

In [ ]:
# テスト
main.main_cli([
    "test",  # run_mode
    "../outputs/logbert/bert_no_time/refine2/fixed/seq_len_512/r_seed_31/weights/ValTotalbest.pth", # 重みファイルまでの相対パス
    "32",   # eval_batchsize
    "cuda:0",     # gpu
    "eval.dataset_dir: ../data/processed/recover/T1105" # テストデータの指定
])

## 可視化
visualize_utils.py を用いる

### ROCグラフ

In [ ]:
visualize_utils.plot_roc(Path("../archive/10/refine2/fixed/seq_len_512/r_seed_31/test_result.csv"))

### TPR-FPRグラフ

In [ ]:
visualize_utils.plot_threshold_metrics(Path("../archive/10/refine2/fixed/seq_len_512/r_seed_31/test_result.csv"))

# 実行例２（書きかけ） 
複数シナリオに対して実行する場合の実行例も示しておきます（時間ありませんでした...）
- 設定したディレクトリに実行例１の応用で、複数のシナリオのデータを作成するイメージ

### ハッシュ値の用意
- 各ログを１つの文字列としてハッシュ関数に通し、ハッシュ値化する
- すべてのシナリオについて一括で変換を行うこと
- 結果は「security_structured_unmasked_{hash_method}」カラムへ格納される

In [ ]:
projects = ["T1105","WEB1","WEB2","T1055","T1007"]
hash_method = "md5"

for project in projects:
    process_csv(
        input_csv_path=INTERIM_DIR/f'{project}/security_structured.csv',
        output_csv_path=INTERIM_DIR/f'{project}/security_structured_unmasked.csv',
        no_mask_config="no_mask.json",
        new_column_name=f'EventTemplateUnmasked',
        hash_method=hash_method,
    )

### モデル前データ作成
- すべてのシナリオについて一括で作成している

In [ ]:
output_dirname = "recover"

eventid = f"EventTemplateUnmaskedHash_{hash_method}"
projects_train = ["T1105"]
projects_test = ["T1105","WEB1","T1055"]
train_ratios = [0.94, 0.7, 0.85]

for projects_train, train_ratios in project, train_ratio:
    preprocess.prepare_model_train_data(
        logdata_filepath = INTERIM_DIR / f"{project}/security_structured_unmasked.csv",
        output_dir = PROCESSED_DIR / output_dirname / project,
        train_ratio=train_ratio,
        features = [f"{eventid}", "deltaT"],
        use_columns = ["timestamp", "Label", "EventID", f"{eventid}", "deltaT"],
        window_size = 430,
        step_size = 200,
        mode = "fixed", 
    )

preprocess.merge_processed_files(
    PROCESSED_DIR/"ex1"/"train", # T1105のtrain
    PROCESSED_DIR/"ex2"/"train", # WEB1のtrain
    PROCESSED_DIR/"tmp/train", # WEB2のtrain
    output_dir=PROCESSED_DIR/"ex3",
    output_filename="train",  # 結合後のファイル名
    mode="fixed",               # seq_stats.txt用のモード
    window_size=430,            # seq_stats.txt用のウィンドウサイズ
    step_size=200,              # seq_stats.txt用のステップサイズ
)
    

for projects, train_ratios in project, train_ratio:
    preprocess.prepare_model_test_data(
        logdata_filepath = INTERIM_DIR / f"{project}/security_structured_unmasked.csv",
        output_dir = PROCESSED_DIR / output_dirname / project,
        train_ratio=train_ratio,
        features = [f"{eventid}", "deltaT"],
        use_columns = ["timestamp", "Label", "EventID", f"{eventid}", "deltaT"],
        window_size = 430,
        step_size = 200,
        mode = "fixed", 
    )

## 実行

In [ ]:
# train
main.main_cli([
    "train",
    "bert/recover",           # conf/bert_config_name.yaml ：configファイルは別途作成しておく
])

## 検証

In [ ]:
# test
main.main_cli([
    "test",  # run_mode
    "../outputs/logbert/bert_no_time/recover/seq_len_512/r_seed_31/weights/ValTotalbest.pth", # モデルの重みファイルまでのパス
    "32",   # eval_batchsize
    "cuda:0",     # gpu
    "eval.dataset_dir=../data/processed/recover/T1055" # 評価を行うテストデータを含むディレクトリまでのパス
])

In [ ]:
visualize_utils.plot_threshold_metrics(Path("../outputs/logbert/bert_no_time/recover/seq_len_512/r_seed_31/test_result.csv"))

In [ ]:
visualize_utils.plot_roc(Path("../outputs/logbert/bert_no_time/recover/seq_len_512/r_seed_31/test_result.csv"))